# PRAM Lab 03 — Strong vs Weak Scaling: from a problem to a scaling experiment

This notebook is meant to answer a practical question:

> **Given a parallel problem, what should I partition, what should stay fixed, and what should I measure?**

We use a **2D 5-point stencil** because it makes decomposition and communication visible.

The algorithmic/cost model is implemented in **C++**. Python is used only for plots and interactive visualizations.

---

## Start with the scientific question, not with the plot

| Your goal | Keep fixed | Increase | Use |
|---|---:|---:|---|
| Finish the **same problem** sooner | global grid side `N` and time horizon `S` | processors `P` | **Strong scaling** |
| Solve a **larger problem** in roughly the same time | local grid side `n` and time horizon `S` | `P` and global grid side `N` | **Weak scaling** |

The two experiments answer different questions. A strong-scaling graph cannot replace a weak-scaling experiment, and vice versa.

> **Important:** `S = number of time steps` is a property of the simulation horizon, not a scaling variable.  
> In the main strong/weak experiments below we keep `S` fixed so that changing `P` is the only scaling experiment.

## 1 · The problem we will partition — notation first

Assume a global **`N × N`** grid and a 5-point stencil. Each time step updates every interior cell from its four neighbors.

We will use four symbols consistently:

$$
\boxed{P=\text{total number of processors}}
$$

$$
\boxed{q=\sqrt P=\text{processors per grid dimension}}
$$

$$
\boxed{N=\text{global grid side}}
\qquad
\boxed{n=\text{local grid side on one processor}}
$$

If processors are arranged as a square `q × q` grid, then

$$
P=q^2.
$$

### Strong scaling

The **global** grid is fixed. Each dimension is divided among `q` processors:

$$
n = \frac{N}{q} = \frac{N}{\sqrt P}.
$$

Therefore the number of cells handled by one processor is

$$
n^2 = \frac{N^2}{P}.
$$

So increasing `P` makes each local subdomain smaller.

### Weak scaling

The **local** grid is fixed:

$$
n=\text{constant}.
$$

With `q=√P` processors per dimension, the global side grows as

$$
N=nq=n\sqrt P.
$$

The global number of cells is therefore

$$
N^2=n^2P.
$$

So increasing `P` increases the global problem while keeping the amount of useful work per processor approximately constant.

### Why communication eventually matters

For a square local subdomain:

- compute grows with its **area**

$$
W_{\text{compute}}\propto n^2;
$$

- halo exchange grows with its **boundary**

$$
W_{\text{halo}}\propto 4n.
$$

Hence

$$
\frac{\text{communication}}{\text{compute}}
\propto
\frac{4n}{n^2}
=
\frac{4}{n}.
$$

This is why small local subdomains are usually less efficient: the boundary becomes large relative to the useful interior work.

In [ ]:

%%bash
# Compile the C++ model.
set -e
cat > stencil_scaling_model.cpp <<'CPP'
#include <cmath>
#include <cstdlib>
#include <iostream>
#include <string>
#include <vector>
using namespace std;

struct Row {
    int P, q, globalN, localN, neighbors;
    long long localCells, haloValues;
    double computeMs, haloMs, latencyMs, syncMs, setupMs, totalMs;
};

static bool square_grid(int P, int &q){
    q = (int) llround(sqrt((double)P));
    return q*q == P;
}

static Row eval_row(const string& mode, int sizeN, int P, int steps, int haloWidth,
                    double computeNs, double boundaryNs, double latencyUs,
                    double syncUs, double setupMs){
    int q=0;
    if(!square_grid(P,q)){ cerr << "P must be a perfect square\n"; exit(2); }
    int globalN, localN;
    if(mode=="strong"){
        globalN=sizeN;
        localN=(globalN + q - 1)/q;
    } else if(mode=="weak"){
        localN=sizeN;
        globalN=localN*q;
    } else { cerr << "unknown mode\n"; exit(2); }

    int neighbors = (q==1 ? 0 : (q==2 ? 2 : 4));
    long long localCells = 1LL*localN*localN;
    long long haloValues = 1LL*neighbors*haloWidth*localN;

    double computeMs = computeNs * 1e-6 * (double)localCells * steps;
    double haloMs = boundaryNs * 1e-6 * (double)haloValues * steps;
    double latencyMs = latencyUs * 1e-3 * neighbors * steps;
    double syncMs = (P==1 ? 0.0 : syncUs * 1e-3 * log2((double)P) * steps);
    double totalMs = setupMs + computeMs + haloMs + latencyMs + syncMs;

    return {P,q,globalN,localN,neighbors,localCells,haloValues,
            computeMs,haloMs,latencyMs,syncMs,setupMs,totalMs};
}

int main(int argc, char** argv){
    if(argc != 11){
        cerr << "usage: stencil_scaling_model MODE SIZE_N STEPS MAX_P HALO COMPUTE_NS BOUNDARY_NS LATENCY_US SYNC_US SETUP_MS\n";
        return 2;
    }
    string mode=argv[1];
    int sizeN=stoi(argv[2]);
    int steps=stoi(argv[3]);
    int maxP=stoi(argv[4]);
    int halo=stoi(argv[5]);
    double computeNs=stod(argv[6]);
    double boundaryNs=stod(argv[7]);
    double latencyUs=stod(argv[8]);
    double syncUs=stod(argv[9]);
    double setupMs=stod(argv[10]);

    vector<int> Ps={1,4,16,64,256};
    cout << "{\"mode\":\""<<mode<<"\",\"rows\":[";
    bool first=true;
    for(int P: Ps){
        if(P>maxP) continue;
        Row r=eval_row(mode,sizeN,P,steps,halo,computeNs,boundaryNs,latencyUs,syncUs,setupMs);
        if(!first) cout << ',';
        first=false;
        cout << "{\"P\":"<<r.P
             <<",\"q\":"<<r.q
             <<",\"globalN\":"<<r.globalN
             <<",\"localN\":"<<r.localN
             <<",\"neighbors\":"<<r.neighbors
             <<",\"localCells\":"<<r.localCells
             <<",\"haloValues\":"<<r.haloValues
             <<",\"computeMs\":"<<r.computeMs
             <<",\"haloMs\":"<<r.haloMs
             <<",\"latencyMs\":"<<r.latencyMs
             <<",\"syncMs\":"<<r.syncMs
             <<",\"setupMs\":"<<r.setupMs
             <<",\"totalMs\":"<<r.totalMs<<"}";
    }
    cout << "]}";
    return 0;
}
CPP

g++ -std=c++17 -O2 stencil_scaling_model.cpp -o stencil_scaling_model
echo "C++ scaling model compiled."


In [ ]:

import json, math, subprocess, sys, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, clear_output, Markdown
if importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

PROFILES = {
    'Fast interconnect': dict(compute_ns=2.0, boundary_ns=5.0, latency_us=0.5, sync_us=0.2, setup_ms=1.0),
    'Balanced':          dict(compute_ns=2.0, boundary_ns=15.0, latency_us=3.0, sync_us=0.8, setup_ms=1.0),
    'Slow interconnect': dict(compute_ns=2.0, boundary_ns=40.0, latency_us=8.0, sync_us=2.0, setup_ms=1.0),
}

def run_model(mode, sizeN, steps, maxP, profile='Balanced', halo=1):
    p = PROFILES[profile]
    cmd = [
        './stencil_scaling_model', mode, str(sizeN), str(steps), str(maxP), str(halo),
        str(p['compute_ns']), str(p['boundary_ns']), str(p['latency_us']),
        str(p['sync_us']), str(p['setup_ms'])
    ]
    return json.loads(subprocess.check_output(cmd, text=True))['rows']

def frame(rows, mode):
    df = pd.DataFrame(rows)
    if mode == 'strong':
        t1 = df.totalMs.iloc[0]
        df['speedup'] = t1 / df.totalMs
        df['efficiency'] = df.speedup / df.P
        df['idealMs'] = t1 / df.P
    else:
        t1 = df.totalMs.iloc[0]
        df['weak_efficiency'] = t1 / df.totalMs
        df['idealMs'] = t1
    df['commMs'] = df.haloMs + df.latencyMs + df.syncMs
    df['comm_share'] = df.commMs / df.totalMs
    return df

print('Visualization helpers ready.')



## 2 · First decision: how should I partition the data?

For a square 2D grid, compare two decompositions:

- **1D strips**: easy to implement, but each rank owns a long boundary;
- **2D blocks**: smaller boundary for the same amount of local work.

For communication-heavy stencil codes, reducing boundary length is often more important than making the partitioning code simpler.


In [ ]:

def draw_partition_compare(globalN=1024, P=16):
    q = int(round(math.sqrt(P)))
    if q*q != P:
        raise ValueError('P must be a perfect square')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))

    # 1D strips
    ax = axes[0]
    ax.add_patch(Rectangle((0,0),1,1,fill=False,lw=2))
    for i in range(1,P):
        ax.plot([i/P,i/P],[0,1],lw=.6,color='#94a3b8')
    sel = min(P//2, P-1)
    ax.add_patch(Rectangle((sel/P,0),1/P,1,facecolor='#dbeafe',edgecolor='#2563eb',lw=1.5))
    ax.set_title('1D strips')
    ax.axis('off'); ax.set_aspect('equal')

    # 2D blocks
    ax = axes[1]
    ax.add_patch(Rectangle((0,0),1,1,fill=False,lw=2))
    for i in range(1,q):
        ax.plot([i/q,i/q],[0,1],lw=.8,color='#94a3b8')
        ax.plot([0,1],[i/q,i/q],lw=.8,color='#94a3b8')
    r=c=q//2
    ax.add_patch(Rectangle((c/q,r/q),1/q,1/q,facecolor='#dcfce7',edgecolor='#16a34a',lw=1.5))
    ax.set_title('2D blocks')
    ax.axis('off'); ax.set_aspect('equal')
    plt.show()

    strip_local_cells = globalN*globalN/P
    strip_halo = 2*globalN
    block_localN = globalN/q
    block_local_cells = block_localN**2
    block_halo = 4*block_localN

    tbl = pd.DataFrame([
        ['1D strips', f'{globalN} × {globalN//P}', int(strip_local_cells), int(strip_halo), strip_halo/strip_local_cells],
        ['2D blocks', f'{globalN//q} × {globalN//q}', int(block_local_cells), int(block_halo), block_halo/block_local_cells],
    ], columns=['Partition','Local shape','Local cells','Approx. halo values / step','Boundary / work'])
    display(tbl)

    ratio = strip_halo / block_halo
    display(Markdown(
        f'**Actionable conclusion:** for `P={P}`, the 2D block decomposition moves about **{ratio:.1f}× less boundary data per rank** than 1D strips in this simple interior-rank model.'
    ))

P_part = widgets.Dropdown(options=[4,16,64,256], value=16, description='P')
N_part = widgets.Dropdown(options=[512,1024,2048,4096], value=1024, description='global N')
out_part = widgets.Output()

def redraw_partition(*_):
    with out_part:
        clear_output(wait=True)
        draw_partition_compare(N_part.value, P_part.value)

P_part.observe(redraw_partition, names='value')
N_part.observe(redraw_partition, names='value')
redraw_partition()
display(widgets.VBox([widgets.HBox([N_part, P_part]), out_part]))


### Cost model used by the notebook

Let

- `S` = number of stencil time steps;
- `n` = local grid side;
- `h` = halo width;
- `P` = total processors;
- `c` = compute cost per cell update;
- `b` = communication cost per halo value;
- `L` = message latency;
- `σ` = synchronization cost;
- `Tsetup` = one-time setup cost.

For one processor, the useful compute work per step is approximately

$$
n^2.
$$

For a processor with four neighbors, the halo size per step is approximately

$$
4hn.
$$

The notebook uses the simplified runtime model

$$
T_P
=
T_{\text{setup}}
+
S\left(
c\,n^2
+
b\,4hn
+
4L
+
\sigma\log_2 P
\right).
$$

This is not intended to predict a particular machine exactly. It is a **reasoning model**: it lets us see which term changes when we change the decomposition.

#### What does `steps = S` do?

`S` multiplies almost every per-step cost:

$$
T_P \approx T_{\text{setup}} + S\cdot T_{\text{per-step}}.
$$

Therefore increasing `S` mostly scales the **absolute runtime** up or down. It does **not** change `N`, `n`, or the partitioning.

In ratios such as strong- or weak-scaling efficiency, the factor `S` largely cancels:

$$
E_{\text{weak}}
=
\frac{T_1}{T_P}
=
\frac{T_{\text{setup}}+S A_1}
     {T_{\text{setup}}+S A_P}.
$$

When setup is small compared with the repeated stencil work,

$$
E_{\text{weak}}\approx\frac{A_1}{A_P},
$$

so changing `S` should **not strongly change the shape of the weak-scaling curve**.

That is why `steps` is kept fixed in the main scaling experiments. We will examine its effect separately later.

## 3 · Strong scaling: “I have this problem. How many processors should I use?”

Strong scaling keeps both the **global problem** and the **simulation horizon** fixed:

$$
N=\text{constant},\qquad S=\text{constant}.
$$

We only increase `P`.

Because

$$
n=\frac{N}{\sqrt P},
$$

the local work decreases as

$$
n^2=\frac{N^2}{P}.
$$

But the halo boundary decreases only as

$$
4n = \frac{4N}{\sqrt P}.
$$

This is the key reason strong scaling eventually saturates:

$$
\text{compute per rank}\sim\frac{1}{P},
\qquad
\text{boundary per rank}\sim\frac{1}{\sqrt P}.
$$

So communication becomes relatively more important as `P` grows.

The actionable question is:

> **At what P do I stop getting a useful return from additional processors?**

In this experiment we keep **`S = 100` steps** fixed. If we changed `S`, absolute runtime would change, but the partitioning logic above would not.

In [ ]:

def plot_strong(globalN=1024, profile='Balanced', maxP=256, efficiency_threshold=0.60, steps=100):
    df = frame(run_model('strong', globalN, steps, maxP, profile), 'strong')

    fig, axes = plt.subplots(1,3,figsize=(15,4.2))

    axes[0].plot(df.P, df.totalMs, marker='o', label='modeled time')
    axes[0].plot(df.P, df.idealMs, marker='o', linestyle='--', label='ideal T1/P')
    axes[0].set_xscale('log',base=4); axes[0].set_yscale('log')
    axes[0].set_xlabel('processors P'); axes[0].set_ylabel('time (ms)'); axes[0].set_title('Time to solution')
    axes[0].grid(True,alpha=.25); axes[0].legend()

    axes[1].plot(df.P, df.efficiency, marker='o')
    axes[1].axhline(efficiency_threshold, linestyle='--', color='gray')
    axes[1].set_xscale('log',base=4); axes[1].set_ylim(0,1.05)
    axes[1].set_xlabel('processors P'); axes[1].set_ylabel('efficiency'); axes[1].set_title('Strong-scaling efficiency')
    axes[1].grid(True,alpha=.25)

    axes[2].plot(df.P, df.localN, marker='o', label='local N')
    axes[2].set_xscale('log',base=4)
    axes[2].set_xlabel('processors P'); axes[2].set_ylabel('local side length'); axes[2].set_title('What partitioning does')
    axes[2].grid(True,alpha=.25)

    plt.tight_layout(); plt.show()

    show = df[['P','globalN','localN','totalMs','speedup','efficiency','comm_share']].copy()
    show['totalMs']=show.totalMs.round(2); show['speedup']=show.speedup.round(2); show['efficiency']=show.efficiency.round(3); show['comm_share']=(100*show.comm_share).round(1)
    show=show.rename(columns={'comm_share':'comm %'})
    display(show)

    ok = df[df.efficiency >= efficiency_threshold]
    p_eff = int(ok.P.max()) if len(ok) else 1
    last = df[df.P==p_eff].iloc[0]
    display(Markdown(
        f'**Decision helper:** with an efficiency threshold of **{efficiency_threshold:.0%}**, the largest tested configuration still above the threshold is **P={p_eff}**. '
        f'At that point `local N={int(last.localN)}`, speedup is **{last.speedup:.1f}×**, and communication/synchronization account for about **{100*last.comm_share:.0f}%** of modeled time.'
    ))


N_strong = widgets.Dropdown(options=[512,1024,2048,4096], value=1024, description='global side N')
profile_strong = widgets.Dropdown(options=list(PROFILES),value='Balanced',description='machine')
maxP_strong = widgets.Dropdown(options=[16,64,256],value=256,description='max P')
eff_strong = widgets.FloatSlider(min=.3,max=.9,step=.05,value=.60,description='min eff.',continuous_update=False)
out_strong = widgets.Output()

STRONG_STEPS = 100

def redraw_strong(*_):
    with out_strong:
        clear_output(wait=True)
        display(Markdown(
            f'**Experiment invariant:** global side $N={N_strong.value}$ and time horizon $S={STRONG_STEPS}$ steps are fixed. '
            f'Only $P$ changes, so $n=N/\\sqrt{{P}}$.'
        ))
        plot_strong(N_strong.value, profile_strong.value, maxP_strong.value, eff_strong.value, STRONG_STEPS)

for w in [N_strong,profile_strong,maxP_strong,eff_strong]:
    w.observe(redraw_strong,names='value')
redraw_strong()
display(widgets.VBox([widgets.HBox([N_strong,profile_strong]), widgets.HBox([maxP_strong,eff_strong]), out_strong]))



### Strong-scaling exercise

Try these two experiments:

1. `global N = 512`, then increase `P`.
2. `global N = 4096`, then increase `P`.

Ask yourself:

- Why does the larger problem tolerate more processors?
- At what point does `local N` become so small that communication is no longer negligible?
- If your objective is a fixed deadline, what is the **smallest P** that meets it?
- If your objective is efficiency, what is the **largest P** you would use?

Strong scaling is a **resource-selection experiment** for a fixed problem.


## 4 · Weak scaling: “I have more processors. Can I solve a bigger problem in the same time?”

Weak scaling keeps the **local problem** and the **simulation horizon** fixed:

$$
n=\text{constant},\qquad S=\text{constant}.
$$

We increase `P`, and therefore the global grid grows:

$$
N=n\sqrt P.
$$

The total number of cells becomes

$$
N^2=n^2P.
$$

### What should remain constant?

For an ideal weak-scaling experiment, useful compute work per processor should remain approximately constant:

$$
W_{\text{compute, rank}}\propto n^2=\text{constant}.
$$

The nearest-neighbor halo per rank also remains approximately constant:

$$
W_{\text{halo, rank}}\propto 4n=\text{constant}.
$$

So if these were the only costs, runtime would be almost flat.

What can still grow with `P`?

- global synchronization;
- collective operations;
- congestion/contention;
- I/O;
- load imbalance.

In our simplified model the synchronization term is

$$
T_{\text{sync}}\propto S\,\log_2 P,
$$

so weak scaling slowly degrades as `P` grows.

The actionable question is:

> **How large a global problem can I solve while keeping the wall-clock penalty acceptable?**

Again, we keep **`S = 100` steps** fixed. `S` changes the duration of the simulation, but it is not the quantity being scaled.

In [ ]:

def plot_weak(localN=256, profile='Balanced', maxP=256, max_slowdown=0.15, steps=100):
    df = frame(run_model('weak', localN, steps, maxP, profile), 'weak')

    fig, axes = plt.subplots(1,3,figsize=(15,4.2))

    axes[0].plot(df.P, df.totalMs, marker='o', label='modeled time')
    axes[0].axhline(df.totalMs.iloc[0], linestyle='--', color='gray', label='ideal flat time')
    axes[0].set_xscale('log',base=4)
    axes[0].set_xlabel('processors P'); axes[0].set_ylabel('time (ms)'); axes[0].set_title('Weak-scaling time')
    axes[0].grid(True,alpha=.25); axes[0].legend()

    axes[1].plot(df.P, df.weak_efficiency, marker='o')
    axes[1].axhline(1/(1+max_slowdown), linestyle='--', color='gray')
    axes[1].set_xscale('log',base=4); axes[1].set_ylim(0,1.05)
    axes[1].set_xlabel('processors P'); axes[1].set_ylabel('T1(local)/TP'); axes[1].set_title('Weak efficiency')
    axes[1].grid(True,alpha=.25)

    axes[2].plot(df.P, df.globalN, marker='o', label='global N')
    axes[2].set_xscale('log',base=4)
    axes[2].set_xlabel('processors P'); axes[2].set_ylabel('global side length'); axes[2].set_title('Problem size you can afford')
    axes[2].grid(True,alpha=.25)

    plt.tight_layout(); plt.show()

    show = df[['P','localN','globalN','localCells','totalMs','weak_efficiency','comm_share']].copy()
    show['totalMs']=show.totalMs.round(2); show['weak_efficiency']=show.weak_efficiency.round(3); show['comm_share']=(100*show.comm_share).round(1)
    show=show.rename(columns={'comm_share':'comm %'})
    display(show)

    limit = df.totalMs.iloc[0]*(1+max_slowdown)
    ok = df[df.totalMs <= limit]
    p_ok = int(ok.P.max()) if len(ok) else 1
    row = df[df.P==p_ok].iloc[0]
    display(Markdown(
        f'**Decision helper:** allowing at most **{max_slowdown:.0%} slowdown** versus one rank, the largest tested configuration is **P={p_ok}**. '
        f'It keeps `local N={localN}` but grows the global grid to **{int(row.globalN)} × {int(row.globalN)}**.'
    ))


n_weak = widgets.Dropdown(options=[64,128,256,512,1024], value=256, description='local side n')
profile_weak = widgets.Dropdown(options=list(PROFILES),value='Balanced',description='machine')
maxP_weak = widgets.Dropdown(options=[16,64,256],value=256,description='max P')
slow_weak = widgets.FloatSlider(min=.05,max=.50,step=.05,value=.15,description='max slowdown',continuous_update=False)
out_weak = widgets.Output()

WEAK_STEPS = 100

def redraw_weak(*_):
    with out_weak:
        clear_output(wait=True)
        display(Markdown(
            f'**Experiment invariant:** local side $n={n_weak.value}$ and time horizon $S={WEAK_STEPS}$ steps are fixed. '
            f'As $P$ grows, the global side grows as $N=n\\sqrt{{P}}$.'
        ))
        plot_weak(n_weak.value, profile_weak.value, maxP_weak.value, slow_weak.value, WEAK_STEPS)

for w in [n_weak,profile_weak,maxP_weak,slow_weak]:
    w.observe(redraw_weak,names='value')
redraw_weak()
display(widgets.VBox([widgets.HBox([n_weak,profile_weak]), widgets.HBox([maxP_weak,slow_weak]), out_weak]))



### Weak-scaling exercise — why `local N` matters

Now change **local N** from `64` to `1024`.

This control is intentionally connected to the model:

- larger local N means more computation per rank (`local N²`);
- boundary work grows only as `local N`;
- therefore larger local subdomains generally hide communication better.

Questions:

- Which `local N` gives better weak efficiency on the same machine profile?
- Why?
- What happens if you switch from **Fast interconnect** to **Slow interconnect**?
- If a weak-scaling run degrades mainly because of synchronization, should you increase local N, reduce synchronization, or both?

Weak scaling is a **capacity experiment**: it tells you how much larger a problem you can attack as resources grow.


### Why changing `steps` barely changes the weak-scaling curve

This is deliberate, not a bug.

For fixed local side `n`, the model can be written as

$$
T_P(S)=T_{\text{setup}}+S\,A_P,
$$

where `A_P` is the cost of **one stencil step** at processor count `P`.

The weak efficiency is

$$
E_{\text{weak}}(P,S)
=
\frac{T_1(S)}{T_P(S)}
=
\frac{T_{\text{setup}}+S A_1}
     {T_{\text{setup}}+S A_P}.
$$

For a long simulation, `S A_P ≫ Tsetup`, hence

$$
E_{\text{weak}}(P,S)\approx\frac{A_1}{A_P}.
$$

So `S` cancels almost completely.

**Interpretation:** changing from 100 to 200 time steps makes the simulation take about twice as long, but it does not fundamentally change whether the decomposition weak-scales well.

The small experiment below makes that distinction explicit.

In [ ]:

def plot_steps_effect(localN=256, P=64, profile='Balanced'):
    steps_values = np.array([10, 20, 50, 100, 200, 500])
    total = []
    weak_eff = []
    for S in steps_values:
        rows = frame(run_model('weak', localN, int(S), P, profile), 'weak')
        rowP = rows[rows.P == P].iloc[0]
        total.append(rowP.totalMs)
        weak_eff.append(rowP.weak_efficiency)

    fig, axes = plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(steps_values, total, marker='o')
    axes[0].set_xlabel('time steps S')
    axes[0].set_ylabel('total runtime (ms)')
    axes[0].set_title('Steps change absolute runtime')
    axes[0].grid(True,alpha=.25)

    axes[1].plot(steps_values, weak_eff, marker='o')
    axes[1].set_xlabel('time steps S')
    axes[1].set_ylabel('weak efficiency')
    axes[1].set_ylim(0,1.05)
    axes[1].set_title('But scaling efficiency changes little')
    axes[1].grid(True,alpha=.25)

    plt.tight_layout()
    plt.show()

plot_steps_effect(localN=256, P=64, profile='Balanced')


## 5 · Combine weak and strong scaling: calibrate locally, then choose P

This is the main conclusion of the lab:

> **Weak scaling determines a useful local granularity; strong scaling uses that granularity to choose `P` for the real problem.**

Think of the two experiments as two phases.

### Phase A — calibrate the machine with weak scaling

Run weak-scaling experiments with several local sides `n`.

For a chosen calibration processor count `P_cal`, define a minimum acceptable weak efficiency

$$
E_{\mathrm{weak}}(P_{\mathrm{cal}},n) \ge E_{\mathrm{weak,min}}.
$$

The **smallest** local side satisfying this condition is our empirical local-granularity limit:

$$
\boxed{n_{\min}=\text{smallest useful local grid side}}
$$

Interpretation:

- if `n < n_min`, each rank is too small and communication/synchronization dominates;
- if `n ≥ n_min`, each rank has enough useful work to amortize communication.

This is **not** a mathematical constant of the algorithm. It depends on the machine, decomposition, halo width, implementation, and efficiency target.

### Phase B — use strong scaling on the real fixed problem

Now suppose the actual problem has global side `N`.

With a square processor grid,

$$
P=q^2,
$$

and the local side is

$$
n(P)=\frac{N}{\sqrt P}.
$$

To avoid making the local problem smaller than the calibrated useful granularity, require

$$
n(P)\ge n_{\min}.
$$

Therefore

$$
\frac{N}{\sqrt P}\ge n_{\min}
$$

and hence

$$
\boxed{
P \le \left(\frac{N}{n_{\min}}\right)^2
}
$$

This gives a **granularity-based upper bound** on the number of processors.

But granularity is not the only condition. We also check the measured/modeled strong-scaling efficiency:

$$
E_{\mathrm{strong}}(P)\ge E_{\mathrm{strong,min}}.
$$

So an actionable choice is

$$
\boxed{
P^\* =
\text{fastest tested configuration satisfying both }
n(P)\ge n_{\min}
\text{ and }
E_{\mathrm{strong}}(P)\ge E_{\mathrm{strong,min}}
}
$$

The widget below performs exactly these two steps.

### How to read the four plots

1. **Weak calibration** — find the smallest useful `n`.
2. **Strong efficiency** — check whether the fixed problem still uses processors efficiently.
3. **Local side** — see when `n(P)` crosses the weak-calibrated `n_min`.
4. **Time to solution** — choose among configurations that satisfy both constraints.

This connects the experiments directly to a resource-allocation decision.

In [ ]:

def combined_scaling_decision(globalN=4096,
                              profile='Balanced',
                              calibrationP=256,
                              weak_eff_min=0.80,
                              strong_eff_min=0.60,
                              maxP=256,
                              steps=100):

    candidate_n = [64, 128, 256, 512, 1024]

    # ---------------------------------------------------------
    # Phase A: weak-scaling calibration of local granularity.
    # ---------------------------------------------------------
    weak_rows = []
    for n in candidate_n:
        dfw = frame(run_model('weak', n, steps, calibrationP, profile), 'weak')
        row = dfw[dfw.P == calibrationP]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        weak_rows.append({
            'localN': n,
            'weak_efficiency': r.weak_efficiency,
            'totalMs': r.totalMs,
            'comm_share': r.comm_share
        })

    weak_cal = pd.DataFrame(weak_rows)
    acceptable_n = weak_cal[weak_cal.weak_efficiency >= weak_eff_min]

    if len(acceptable_n):
        n_min = int(acceptable_n.localN.min())
        calibration_ok = True
    else:
        n_min = int(weak_cal.localN.max())
        calibration_ok = False

    # ---------------------------------------------------------
    # Phase B: strong scaling of the real fixed global problem.
    # ---------------------------------------------------------
    dfs = frame(run_model('strong', globalN, steps, maxP, profile), 'strong').copy()
    dfs['granularity_ok'] = dfs.localN >= n_min
    dfs['strong_eff_ok'] = dfs.efficiency >= strong_eff_min
    dfs['valid'] = dfs.granularity_ok & dfs.strong_eff_ok

    # Continuous granularity bound; actual experiment still uses only tested P.
    p_granularity_bound = (globalN / n_min) ** 2

    valid = dfs[dfs.valid]
    if len(valid):
        # Pick the fastest tested configuration satisfying BOTH conditions.
        chosen = valid.loc[valid.totalMs.idxmin()]
        p_star = int(chosen.P)
    else:
        chosen = dfs.iloc[0]
        p_star = 1

    fastest_overall = dfs.loc[dfs.totalMs.idxmin()]

    # ---------------------------------------------------------
    # Visualization.
    # ---------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # 1) Weak calibration.
    ax = axes[0,0]
    ax.plot(weak_cal.localN, weak_cal.weak_efficiency, marker='o')
    ax.axhline(weak_eff_min, linestyle='--', color='gray',
               label=f'min weak efficiency = {weak_eff_min:.0%}')
    ax.axvline(n_min, linestyle=':', color='black',
               label=f'n_min = {n_min}')
    ax.set_xscale('log', base=2)
    ax.set_ylim(0,1.05)
    ax.set_xlabel('local side n')
    ax.set_ylabel('weak efficiency')
    ax.set_title(f'A. Weak calibration at P_cal={calibrationP}')
    ax.grid(True, alpha=.25)
    ax.legend()

    # 2) Strong efficiency.
    ax = axes[0,1]
    ax.plot(dfs.P, dfs.efficiency, marker='o')
    ax.axhline(strong_eff_min, linestyle='--', color='gray',
               label=f'min strong efficiency = {strong_eff_min:.0%}')
    if len(valid):
        ax.scatter([p_star], [chosen.efficiency], s=110, zorder=5,
                   label=f'chosen P*={p_star}')
    ax.set_xscale('log', base=4)
    ax.set_ylim(0,1.05)
    ax.set_xlabel('processors P')
    ax.set_ylabel('strong efficiency')
    ax.set_title('B. Strong scaling of the fixed problem')
    ax.grid(True, alpha=.25)
    ax.legend()

    # 3) Local side versus P.
    ax = axes[1,0]
    ax.plot(dfs.P, dfs.localN, marker='o', label='n(P) = N / sqrt(P)')
    ax.axhline(n_min, linestyle='--', color='gray',
               label=f'weak-calibrated n_min={n_min}')
    bad = dfs[~dfs.granularity_ok]
    if len(bad):
        ax.scatter(bad.P, bad.localN, marker='x', s=90,
                   label='below useful granularity')
    if len(valid):
        ax.scatter([p_star], [chosen.localN], s=110, zorder=5)
    ax.set_xscale('log', base=4)
    ax.set_yscale('log', base=2)
    ax.set_xlabel('processors P')
    ax.set_ylabel('local side n(P)')
    ax.set_title('C. Does strong scaling make ranks too small?')
    ax.grid(True, alpha=.25)
    ax.legend()

    # 4) Time and validity.
    ax = axes[1,1]
    ax.plot(dfs.P, dfs.totalMs, marker='o', label='time to solution')
    invalid = dfs[~dfs.valid]
    if len(invalid):
        ax.scatter(invalid.P, invalid.totalMs, marker='x', s=90,
                   label='reject: granularity and/or efficiency')
    if len(valid):
        ax.scatter([p_star], [chosen.totalMs], s=130, zorder=6,
                   label=f'P*={p_star}')
    ax.set_xscale('log', base=4)
    ax.set_yscale('log')
    ax.set_xlabel('processors P')
    ax.set_ylabel('modeled time (ms)')
    ax.set_title('D. Resource decision')
    ax.grid(True, alpha=.25)
    ax.legend()

    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # Tables and decision statement.
    # ---------------------------------------------------------
    cal_show = weak_cal.copy()
    cal_show['weak_efficiency'] = cal_show.weak_efficiency.round(3)
    cal_show['comm %'] = (100*cal_show.comm_share).round(1)
    cal_show = cal_show[['localN','weak_efficiency','comm %']]
    display(Markdown('### Phase A — weak-scaling calibration'))
    display(cal_show)

    strong_show = dfs[['P','q','globalN','localN','totalMs','efficiency',
                       'granularity_ok','strong_eff_ok','valid']].copy()
    strong_show['totalMs'] = strong_show.totalMs.round(2)
    strong_show['efficiency'] = strong_show.efficiency.round(3)
    display(Markdown('### Phase B — strong-scaling deployment'))
    display(strong_show)

    if calibration_ok:
        calibration_msg = (
            f'Weak scaling says the smallest tested useful local side is **n_min={n_min}** '
            f'at **P_cal={calibrationP}**, using the target '
            f'**E_weak ≥ {weak_eff_min:.0%}**.'
        )
    else:
        calibration_msg = (
            f'No tested local side reached **E_weak ≥ {weak_eff_min:.0%}** at '
            f'**P_cal={calibrationP}**. The displayed n_min={n_min} is only the largest '
            'candidate tested; expand the calibration range.'
        )

    if len(valid):
        decision_msg = (
            f'For the real problem **N={globalN}**, the weak-calibrated granularity gives '
            f'the continuous bound **P ≤ (N/n_min)² ≈ {p_granularity_bound:.1f}**. '
            f'After also requiring **E_strong ≥ {strong_eff_min:.0%}**, the fastest tested '
            f'valid configuration is **P*={p_star}** (`q={int(chosen.q)}`, '
            f'`local side n={int(chosen.localN)}`), with modeled strong efficiency '
            f'**{chosen.efficiency:.1%}**.'
        )
    else:
        decision_msg = (
            'No tested configuration satisfies both the local-granularity and '
            'strong-efficiency constraints. Use fewer processors, increase N, '
            'or revisit the decomposition/communication cost.'
        )

    extra = ''
    if int(fastest_overall.P) != p_star:
        extra = (
            f' The absolute fastest tested point is P={int(fastest_overall.P)}, '
            'but it is rejected by at least one of the two constraints.'
        )

    display(Markdown(
        '**Combined conclusion.** ' + calibration_msg + '  \n\n' +
        decision_msg + extra
    ))


# Interactive combined experiment.
N_combined = widgets.Dropdown(
    options=[1024, 2048, 4096, 8192],
    value=4096,
    description='global side N'
)
profile_combined = widgets.Dropdown(
    options=list(PROFILES),
    value='Balanced',
    description='machine'
)
pcal_combined = widgets.Dropdown(
    options=[16, 64, 256],
    value=256,
    description='calibration P'
)
weak_eff_combined = widgets.FloatSlider(
    min=.50, max=.95, step=.05, value=.80,
    description='min weak eff.',
    continuous_update=False
)
strong_eff_combined = widgets.FloatSlider(
    min=.30, max=.90, step=.05, value=.60,
    description='min strong eff.',
    continuous_update=False
)
maxp_combined = widgets.Dropdown(
    options=[16, 64, 256],
    value=256,
    description='max strong P'
)
out_combined = widgets.Output()

COMBINED_STEPS = 100

def redraw_combined(*_):
    with out_combined:
        clear_output(wait=True)
        display(Markdown(
            f'**Calibration → deployment:** first infer a useful local side from weak scaling '
            f'at `P_cal={pcal_combined.value}`; then use it to constrain strong scaling of '
            f'the fixed `N={N_combined.value}` problem. `S={COMBINED_STEPS}` steps stays fixed.'
        ))
        combined_scaling_decision(
            globalN=N_combined.value,
            profile=profile_combined.value,
            calibrationP=pcal_combined.value,
            weak_eff_min=weak_eff_combined.value,
            strong_eff_min=strong_eff_combined.value,
            maxP=maxp_combined.value,
            steps=COMBINED_STEPS
        )

for w in [N_combined, profile_combined, pcal_combined,
          weak_eff_combined, strong_eff_combined, maxp_combined]:
    w.observe(redraw_combined, names='value')

redraw_combined()

display(widgets.VBox([
    widgets.HBox([N_combined, profile_combined]),
    widgets.HBox([pcal_combined, maxp_combined]),
    widgets.HBox([weak_eff_combined, strong_eff_combined]),
    out_combined
]))


## 6 · A practical workflow for a new parallel problem

When students meet a new problem, use this sequence.

1. **Identify the data domain.** Vector, grid, graph, matrix, particles, independent tasks?
2. **Choose a decomposition.** Minimize cross-partition dependencies, not only the number of elements per rank.
3. **Estimate local work and boundary work.** For a 2D grid, compare area `n²` with boundary `4n`.
4. **Fix the scientific problem definition.** In this example, keep the simulation horizon `S` fixed while testing scaling.
5. **Calibrate useful local granularity with weak scaling.**
   - keep local side `n` fixed;
   - increase `P`;
   - determine the smallest `n` that preserves an acceptable weak efficiency;
   - call it `n_min`.
6. **Apply that calibration to the real fixed problem with strong scaling.**
   - global side `N` is fixed;
   - compute `n(P)=N/√P`;
   - reject configurations with `n(P)<n_min`.
7. **Also enforce a strong-efficiency target.**
   - granularity alone is not enough;
   - communication, synchronization, and setup can still make a configuration inefficient.
8. **Choose P using both constraints.**
   - weak scaling tells you whether each rank has enough work;
   - strong scaling tells you whether adding ranks still helps the actual fixed problem.
9. **Inspect the time breakdown.** Compute, halo transfer, latency, synchronization, I/O.
10. **Document the decision.** Report `N`, `P`, `q=√P`, local side `n`, efficiency thresholds, and why the selected point is acceptable.

The core design pattern is:

> **weak scaling calibrates the useful local granularity; strong scaling uses that calibration to choose resources for the real problem.**

## 7 · Final challenge — make one resource decision

Use the **combined scaling** widget above.

### Scenario

You have a fixed `4096 × 4096` stencil problem and the **Balanced** machine profile.

1. Calibrate local granularity at `P_cal = 256`.
2. Require at least **80% weak efficiency**.
3. Record the resulting `n_min`.
4. For the real fixed problem, require at least **60% strong efficiency**.
5. Determine the tested `P*` selected by the combined decision.

Then explain the result using the following chain:

$$
\text{weak scaling}
\rightarrow
n_{\min}
\rightarrow
n(P)=\frac{N}{\sqrt P}
\rightarrow
\text{granularity constraint}
\rightarrow
\text{strong-scaling check}
\rightarrow
P^\*
$$

### Questions to answer

- What is `n_min`, and why did weak scaling determine it?
- What continuous processor bound follows from

$$
P \le \left(\frac{N}{n_{\min}}\right)^2?
$$

- Which tested configurations are rejected because `n(P) < n_min`?
- Which are rejected because strong efficiency is too low?
- Is the fastest tested configuration necessarily the configuration you should choose?
- How does the answer change for the **Slow interconnect** profile?

The desired conclusion is not merely a processor count. It is a defensible resource decision:

> **We first learned how much work a rank needs from weak scaling; then we used that local-granularity requirement to interpret strong scaling of the actual fixed problem.**